In [5]:
from transformers import CLIPProcessor, CLIPModel
import torch
from torch.utils.data import DataLoader
from torchvision.datasets import Imagenette
from sklearn.metrics import accuracy_score
import numpy as np
import torchvision
from tqdm import tqdm

In [6]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [7]:
def custom_collate_function(batch):
  images = [item[0] for item in batch]
  labels = [item[1] for item in batch]
  labels = torch.tensor(labels, dtype=torch.long)
  return images, labels

In [8]:
def ValidateModel(model, device, data_loader, text_features):
  
  model.eval()
  true_labels = torch.tensor([]).to(device)
  predicted_labels = torch.tensor([]).to(device)

  text_features = text_features / text_features.norm(dim=-1, keepdim=True)
  logit_scale = model.logit_scale.exp()

  with torch.no_grad():
    for images, labels in tqdm(data_loader):
      labels = labels.to(device)
      image_inputs = processor(images = images, return_tensors = "pt", padding = True).to(device)
      image_features = model.get_image_features(**image_inputs)
       # Косинусное сходство между image features и text features + масштабирование
      similarity = (image_features @ text_features.T) * logit_scale

      predicted = similarity.argmax(dim=1)
      true_labels = torch.cat((true_labels, labels), 0)
      predicted_labels = torch.cat((predicted_labels, predicted), 0)

  return true_labels, predicted_labels

In [9]:
import os
from sklearn.metrics import f1_score
os.environ["TOKENIZERS_PARALLELISM"] = "false"

val_transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize((224, 224)),
])
val_dataset = Imagenette(root='./data', split='val', download=True, transform = val_transform)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=7, collate_fn = custom_collate_function)

all_classes = [label[0] for label in val_dataset.classes]
text_inputs = processor(text = all_classes, return_tensors = "pt", padding = True).to(DEVICE)

model = model.to(DEVICE)
model.eval()

text_features = model.get_text_features(**text_inputs)

In [10]:
# Замеряем при 224x224
val_transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize((224, 224)),
])
val_dataset = Imagenette(root='./data', split='val', download=True, transform = val_transform)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=7, collate_fn = custom_collate_function)

true_data, predicted_data = ValidateModel(model, device = DEVICE, data_loader = val_loader, text_features = text_features)
f1_res = f1_score(true_data.cpu().numpy(), predicted_data.cpu().numpy(), average = 'macro')

print("F1 Score: ", f1_res)

100%|██████████| 123/123 [00:34<00:00,  3.52it/s]

F1 Score:  0.9876700606902598


In [ ]:
# Не меняем размеры
val_transform = torchvision.transforms.Compose([
])
val_dataset = Imagenette(root='./data', split='val', download=True, transform = val_transform)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=7, collate_fn = custom_collate_function)

true_data, predicted_data = ValidateModel(model, device = DEVICE, data_loader = val_loader, text_features = text_features)
f1_res = f1_score(true_data.cpu().numpy(), predicted_data.cpu().numpy(), average = 'macro')

print("F1 Score: ", f1_res)

100%|██████████| 123/123 [00:49<00:00,  2.46it/s]

F1 Score:  0.9884125284251883


In [12]:
# Ставим 10x10
val_transform = torchvision.transforms.Compose([
  torchvision.transforms.Resize((10, 10)),
])
val_dataset = Imagenette(root='./data', split='val', download=True, transform = val_transform)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=7, collate_fn = custom_collate_function)

true_data, predicted_data = ValidateModel(model, device = DEVICE, data_loader = val_loader, text_features = text_features)
f1_res = f1_score(true_data.cpu().numpy(), predicted_data.cpu().numpy(), average = 'macro')

print("F1 Score: ", f1_res)

100%|██████████| 123/123 [00:36<00:00,  3.41it/s]

F1 Score:  0.3424756890373483


In [13]:
# Ставим 1000x1000
val_transform = torchvision.transforms.Compose([
  torchvision.transforms.Resize((1000, 1000)),
])
val_dataset = Imagenette(root='./data', split='val', download=True, transform = val_transform)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=7, collate_fn = custom_collate_function)

true_data, predicted_data = ValidateModel(model, device = DEVICE, data_loader = val_loader, text_features = text_features)
f1_res = f1_score(true_data.cpu().numpy(), predicted_data.cpu().numpy(), average = 'macro')

print("F1 Score: ", f1_res)

100%|██████████| 123/123 [01:58<00:00,  1.04it/s]

F1 Score:  0.9871516375806003
